# 📘 Архитектуры Агентов 18: Plan-Execute-Observe-Replan (Динамическое Перепланирование)

В этом блокноте мы исследуем архитектуру **Plan-Execute-Observe-Replan** — мощный паттерн, который выводит адаптивность агента на новый уровень. В отличие от архитектуры PEV (Plan-Execute-Verify), где Верификатор просто проверяет успех/провал, здесь мы добавляем **Observer** (Наблюдатель), который проводит глубокий анализ результатов, и **Replanner** (Перепланировщик), способный полностью пересмотреть стратегию.

Ключевое отличие этой архитектуры — способность не просто повторять неудавшийся шаг, а **полностью переосмыслить подход** к решению задачи на основе полученных наблюдений. Это особенно ценно при работе с ненадежными инструментами, динамически меняющимися данными или задачами, где первоначальный план оказывается неоптимальным.

Для демонстрации мы создадим набор "ломающихся" инструментов, которые симулируют реальные проблемы: частичные данные, таймауты, устаревшую информацию. Затем мы сравним поведение базового PEV агента и нашего Replan-агента.

### Определение
Архитектура **Plan-Execute-Observe-Replan** — это расширенный цикл управления агентом, где каждое действие не только проверяется на успешность, но и анализируется на предмет полученных инсайтов. Наблюдатель извлекает информацию, которая может повлиять на дальнейшую стратегию, а Перепланировщик адаптирует план на основе этих наблюдений.

### Высокоуровневый рабочий процесс

1.  **План (Plan):** Агент создает начальный план для достижения цели.
2.  **Выполнение (Execute):** Исполнитель выполняет следующий шаг плана.
3.  **Наблюдение (Observe):** Наблюдатель анализирует результат:
    *   Извлекает ключевые факты и данные
    *   Определяет, соответствует ли результат ожиданиям
    *   Выявляет новую информацию, влияющую на план
4.  **Перепланирование (Replan):** На основе наблюдений принимается решение:
    *   **Продолжить** текущий план
    *   **Модифицировать** план (добавить/удалить шаги)
    *   **Полностью пересмотреть** стратегию

### Когда использовать / Применения
*   **Динамические среды:** Когда данные или условия меняются во время выполнения.
*   **Исследовательские задачи:** Когда неизвестно заранее, какой путь приведет к успеху.
*   **Ненадежные инструменты:** При работе с API, которые могут возвращать неполные или устаревшие данные.
*   **Сложные многошаговые задачи:** Где результат одного шага существенно влияет на следующие.

### Сильные и слабые стороны
*   **Сильные стороны:**
    *   **Высокая адаптивность:** Способность менять стратегию "на лету".
    *   **Глубокий анализ:** Наблюдатель извлекает больше информации, чем простая проверка.
    *   **Устойчивость к неопределенности:** Работает даже когда начальный план неоптимален.
*   **Слабые стороны:**
    *   **Увеличенная стоимость:** Больше вызовов LLM для анализа и перепланирования.
    *   **Риск бесконечных циклов:** Необходимы ограничения на количество перепланирований.
    *   **Сложность отладки:** Динамически меняющиеся планы труднее отслеживать.

## Фаза 0: Основа и Настройка

Мы начнем с установки наших библиотек и настройки API ключей.

### Шаг 0.1: Установка библиотек

**Что мы будем делать:**
Мы установим необходимые библиотеки: `langchain`, `langgraph` для логики агентов, `rich` для красивого вывода и `langchain-nebius` для доступа к моделям.

In [ ]:
# !pip install -q -U langchain-nebius langchain langgraph rich python-dotenv langchain-tavily

### Шаг 0.2: Импорт и настройка окружения

**Что мы будем делать:**
1. Импортируем классы для графов, моделей и промптов.
2. Загрузим API ключи из `.env` файла.
3. Настроим трассировку LangSmith для отладки.
4. Инициализируем Rich Console для цветного вывода логов.

In [ ]:
import os
import json
import random
import time
from typing import List, Annotated, TypedDict, Optional, Literal, Dict, Any
from dotenv import load_dotenv

# Pydantic для моделирования данных
from pydantic import BaseModel, Field

# Компоненты LangChain
from langchain_nebius import ChatNebius
from langchain_community.tools.tavily_search import TavilySearchResults as TavilySearch
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

# Компоненты LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import AnyMessage, add_messages

# Для красивого вывода
from rich.console import Console
from rich.markdown import Markdown
from rich.panel import Panel
from rich.table import Table

from notebook_utils import setup_environment

# --- Настройка API ключей и трассировки ---
console = setup_environment(
    project_name="Agentic Architecture - Replan Dynamic (Nebius)",
    required_keys=["NEBIUS_API_KEY", "LANGCHAIN_API_KEY", "TAVILY_API_KEY"]
)

### Шаг 0.3: Определение "Ломающихся" Инструментов

**Что мы будем делать:**
Мы создадим специальные версии инструментов, которые намеренно работают с ошибками. Это необходимо, чтобы протестировать способность агента восстанавливаться.

**Типы сбоев:**
1. **`flaky_search_tool`**: 
   - Иногда возвращает частичные данные.
   - Иногда выдает ошибку таймаута.
   - Может вернуть устаревшие данные.
2. **`flaky_data_tool`**:
   - Может вернуть данные в неверном формате (XML вместо JSON).
3. **`flaky_calculate_tool`**:
   - Требует уточнений, если не хватает данных для расчета.

In [ ]:
console = Console()
llm = ChatNebius(model="meta-llama/Meta-Llama-3.1-8B-Instruct", temperature=0)

# Счетчик вызовов для симуляции разного поведения
tool_call_counts = {"search": 0, "data": 0, "calculate": 0}

def flaky_search_tool(query: str) -> str:
    """Поисковый инструмент, который иногда возвращает неполные результаты."""
    tool_call_counts["search"] += 1
    console.print(f"--- TOOL [flaky_search]: Поиск '{query}'... (вызов #{tool_call_counts['search']}) ---")
    
    # Симуляция различных сбоев
    if "competitor" in query.lower() and tool_call_counts["search"] == 1:
        console.print("--- TOOL: [bold yellow]Возврат неполных данных![/bold yellow] ---")
        return "PARTIAL_DATA: Found information about 2 of 5 competitors. Main competitor: CompanyX with 30% market share. [Data incomplete due to API limitations]"
    
    if "revenue" in query.lower() and tool_call_counts["search"] <= 2:
        console.print("--- TOOL: [bold red]Таймаут API![/bold red] ---")
        return "ERROR: Request timeout after 30s. The financial data service is temporarily unavailable. Suggest trying alternative source."
    
    if "market" in query.lower() and tool_call_counts["search"] == 1:
        console.print("--- TOOL: [bold yellow]Устаревшие данные![/bold yellow] ---")
        return "STALE_DATA (from 2023-Q2): Market size was $50B. WARNING: This data is 18 months old. Current estimates may differ by 20-30%."
    
    # Успешный результат с Tavily
    try:
        result = TavilySearch(max_results=2).invoke(query)
        if isinstance(result, (dict, list)):
            return json.dumps(result, indent=2)
        return str(result)
    except Exception as e:
        return f"Search completed. Found relevant information about: {query}"

def flaky_data_tool(data_type: str) -> str:
    """Инструмент получения данных с различными типами сбоев."""
    tool_call_counts["data"] += 1
    console.print(f"--- TOOL [flaky_data]: Запрос данных '{data_type}'... (вызов #{tool_call_counts['data']}) ---")
    
    if "employee" in data_type.lower():
        if tool_call_counts["data"] == 1:
            console.print("--- TOOL: [bold red]Ошибка формата данных![/bold red] ---")
            return "FORMAT_ERROR: Expected JSON but received XML. Raw data: <employees><count>unknown</count></employees>"
        else:
            return "Employee count: 15,000 (as of Q4 2024)"
    
    if "financial" in data_type.lower():
        return json.dumps({
            "revenue": "$25.5B",
            "growth": "15% YoY",
            "profit_margin": "22%"
        })
    
    return f"Data retrieved for: {data_type}"

def flaky_calculate_tool(expression: str) -> str:
    """Калькулятор, который иногда требует уточнения."""
    tool_call_counts["calculate"] += 1
    console.print(f"--- TOOL [flaky_calculate]: Вычисление '{expression}'... ---")
    
    if "per employee" in expression.lower() and "revenue" not in expression.lower():
        return "CLARIFICATION_NEEDED: Cannot calculate 'per employee' metric. Missing revenue or other base value. Please provide the numerator."
    
    # Простой калькулятор
    try:
        # Извлекаем числа для простого расчета
        numbers = [float(x) for x in expression.split() if x.replace('.', '').replace(',', '').isdigit()]
        if len(numbers) >= 2:
            return f"Calculation result: {numbers[0] / numbers[1]:.2f}"
    except:
        pass
    
    return f"Processed calculation: {expression}"

# Сброс счетчиков
def reset_tool_counters():
    global tool_call_counts
    tool_call_counts = {"search": 0, "data": 0, "calculate": 0}

print("Ломающиеся инструменты определены.")

## Фаза 1: Базовый уровень — Агент PEV

Сначала мы создадим упрощенный PEV (Plan-Execute-Verify) агент для сравнения. Этот агент имеет простую логику верификации: проверяет, содержит ли результат ошибку, и либо продолжает, либо повторяет тот же шаг.

### Шаг 1.1: Реализация структуры PEV

**Что мы будем делать:**
1. Определим состояние `PEVState` для хранения плана и промежуточных данных.
2. Создадим узлы графа:
   - **`pev_planner_node`**: Создает линейный план шагов.
   - **`pev_executor_node`**: Выполняет текущий шаг.
   - **`pev_verifier_node`**: Проверяет успешность выполнения (Success/Retry).
   - **`pev_router`**: Определяет следующий шаг (Retry, Execute Next, или Finish).
3. Соберем граф `pev_agent`.

In [ ]:
# Pydantic модели для структурированного вывода
class Plan(BaseModel):
    steps: List[str] = Field(description="Список шагов для выполнения")

class VerificationResult(BaseModel):
    is_successful: bool = Field(description="Успешен ли шаг")
    should_retry: bool = Field(description="Нужно ли повторить шаг")
    reasoning: str = Field(description="Обоснование решения")

class PEVState(TypedDict):
    user_request: str
    plan: Optional[List[str]]
    current_step_index: int
    last_result: Optional[str]
    collected_data: List[str]
    final_answer: Optional[str]
    retry_count: int
    max_retries: int

def pev_planner_node(state: PEVState):
    console.print("--- [PEV] PLANNER: Создание плана... ---")
    planner_llm = llm.with_structured_output(Plan)
    
    prompt = f"""
    Create a plan to answer this request: "{state['user_request']}"
    
    Available tools:
    - flaky_search_tool(query): Search for information
    - flaky_data_tool(data_type): Get specific data
    - flaky_calculate_tool(expression): Perform calculations
    
    Return a list of 3-5 specific steps using these tools.
    """
    
    plan = planner_llm.invoke(prompt)
    console.print(f"[cyan]План: {plan.steps}[/cyan]")
    return {"plan": plan.steps, "current_step_index": 0, "collected_data": []}

def pev_executor_node(state: PEVState):
    step_idx = state["current_step_index"]
    step = state["plan"][step_idx]
    console.print(f"--- [PEV] EXECUTOR: Выполнение шага {step_idx + 1}: '{step}' ---")
    
    # Определяем, какой инструмент использовать
    if "search" in step.lower() or "find" in step.lower():
        result = flaky_search_tool(step)
    elif "data" in step.lower() or "get" in step.lower():
        result = flaky_data_tool(step)
    elif "calcul" in step.lower() or "compute" in step.lower():
        result = flaky_calculate_tool(step)
    else:
        result = flaky_search_tool(step)
    
    return {"last_result": result}

def pev_verifier_node(state: PEVState):
    console.print("--- [PEV] VERIFIER: Проверка результата... ---")
    verifier_llm = llm.with_structured_output(VerificationResult)
    
    prompt = f"""
    Verify this tool result:
    Step: {state['plan'][state['current_step_index']]}
    Result: {state['last_result']}
    
    Determine if the step was successful or if it should be retried.
    Look for ERROR, TIMEOUT, PARTIAL_DATA, FORMAT_ERROR keywords.
    """
    
    verification = verifier_llm.invoke(prompt)
    console.print(f"--- [PEV] VERIFIER: {'✓ Успех' if verification.is_successful else '✗ Требуется повтор'} ---")
    
    if verification.is_successful:
        new_data = state["collected_data"] + [state["last_result"]]
        return {
            "collected_data": new_data,
            "current_step_index": state["current_step_index"] + 1,
            "retry_count": 0
        }
    else:
        return {"retry_count": state["retry_count"] + 1}

def pev_synthesizer_node(state: PEVState):
    console.print("--- [PEV] SYNTHESIZER: Генерация ответа... ---")
    
    context = "\n".join(state["collected_data"])
    prompt = f"""
    Synthesize an answer for: "{state['user_request']}"
    
    Collected data:
    {context}
    
    Provide a comprehensive answer based on the available data.
    Note any limitations if data was incomplete.
    """
    
    answer = llm.invoke(prompt).content
    return {"final_answer": answer}

def pev_router(state: PEVState):
    # Проверка на превышение лимита retry
    if state["retry_count"] >= state["max_retries"]:
        console.print("--- [PEV] ROUTER: Лимит повторов достигнут, переход к синтезу... ---")
        # Добавляем частичные данные и двигаемся дальше
        return "skip_step"
    
    if state["retry_count"] > 0:
        console.print(f"--- [PEV] ROUTER: Повтор шага (попытка {state['retry_count'] + 1})... ---")
        return "retry"
    
    if state["current_step_index"] >= len(state["plan"]):
        console.print("--- [PEV] ROUTER: План завершен, переход к синтезу ---")
        return "synthesize"
    
    console.print("--- [PEV] ROUTER: Продолжение выполнения ---")
    return "continue"

def pev_skip_step_node(state: PEVState):
    console.print("--- [PEV] SKIP: Пропуск шага с частичными данными ---")
    new_data = state["collected_data"] + [f"[SKIPPED with partial data: {state['last_result']}]"]
    return {
        "collected_data": new_data,
        "current_step_index": state["current_step_index"] + 1,
        "retry_count": 0
    }

# Построение графа PEV
pev_graph = StateGraph(PEVState)
pev_graph.add_node("plan", pev_planner_node)
pev_graph.add_node("execute", pev_executor_node)
pev_graph.add_node("verify", pev_verifier_node)
pev_graph.add_node("skip_step", pev_skip_step_node)
pev_graph.add_node("synthesize", pev_synthesizer_node)

pev_graph.set_entry_point("plan")
pev_graph.add_edge("plan", "execute")
pev_graph.add_edge("execute", "verify")
pev_graph.add_conditional_edges("verify", pev_router, {
    "continue": "execute",
    "retry": "execute",
    "skip_step": "skip_step",
    "synthesize": "synthesize"
})
pev_graph.add_conditional_edges("skip_step", lambda s: "synthesize" if s["current_step_index"] >= len(s["plan"]) else "execute")
pev_graph.add_edge("synthesize", END)

pev_agent = pev_graph.compile()
print("Базовый PEV агент скомпилирован.")

## Фаза 2: Продвинутый подход — Агент Replan Dynamic

Теперь мы построим полноценного Replan-агента с:
- **Observer** — глубокий анализ результатов с извлечением инсайтов
- **Replanner** — умное перепланирование на основе наблюдений

### Шаг 2.1: Реализация узлов Replan агента

**Что мы будем делать:**
1. Создадим модель `Observation` для глубокого анализа результатов (факты, проблемы, предложения).
2. Создадим модель `ReplanDecision` для выбора стратегии (Continue, Modify, Full Replan).
3. Реализуем узлы:
   - **`replan_observer_node`**: Анализирует вывод инструмента, выявляет "ломающиеся" паттерны.
   - **`replan_replanner_node`**: Принимает решение об изменении плана на основе наблюдений.
   - **`replan_executor_node`**: Исполняет шаги и сохраняет полную историю (шаг + результат).

In [ ]:
# Pydantic модели для Replan агента
class Observation(BaseModel):
    status: Literal["success", "partial", "failure", "needs_alternative"] = Field(
        description="Статус выполнения шага"
    )
    extracted_facts: List[str] = Field(
        description="Извлеченные факты из результата"
    )
    issues_found: List[str] = Field(
        description="Обнаруженные проблемы"
    )
    suggestions: List[str] = Field(
        description="Предложения по улучшению плана"
    )

class ReplanDecision(BaseModel):
    action: Literal["continue", "modify_plan", "full_replan", "synthesize"] = Field(
        description="Решение о дальнейших действиях"
    )
    new_steps: Optional[List[str]] = Field(
        default=None,
        description="Новые или модифицированные шаги (если применимо)"
    )
    reasoning: str = Field(
        description="Обоснование решения"
    )

class ReplanState(TypedDict):
    user_request: str
    current_plan: List[str]
    current_step_index: int
    execution_history: List[Dict[str, Any]]  # {step, result, observation}
    observations: List[str]
    extracted_facts: List[str]
    final_answer: Optional[str]
    replan_count: int
    max_replans: int

def replan_planner_node(state: ReplanState):
    console.print("--- [REPLAN] PLANNER: Создание начального плана... ---")
    planner_llm = llm.with_structured_output(Plan)
    
    # Учитываем предыдущие попытки, если они были
    history_context = ""
    if state.get("execution_history"):
        history_context = f"\nПредыдущие попытки и их результаты:\n{json.dumps(state['execution_history'], indent=2)}"
    
    prompt = f"""
    Create a plan to answer: "{state['user_request']}"
    {history_context}
    
    Available tools:
    - flaky_search_tool(query): Search for information (may return partial data)
    - flaky_data_tool(data_type): Get specific data (may have format issues)
    - flaky_calculate_tool(expression): Perform calculations
    
    Create a robust plan with 3-5 steps. Consider using alternative approaches if primary ones might fail.
    """
    
    plan = planner_llm.invoke(prompt)
    console.print(Panel("\n".join([f"{i+1}. {s}" for i, s in enumerate(plan.steps)]), title="План", border_style="cyan"))
    
    return {
        "current_plan": plan.steps,
        "current_step_index": 0,
        "execution_history": state.get("execution_history", []),
        "extracted_facts": state.get("extracted_facts", [])
    }

def replan_executor_node(state: ReplanState):
    step_idx = state["current_step_index"]
    step = state["current_plan"][step_idx]
    console.print(f"--- [REPLAN] EXECUTOR: Шаг {step_idx + 1}/{len(state['current_plan'])}: '{step}' ---")
    
    # Выбор инструмента
    if "search" in step.lower() or "find" in step.lower() or "look" in step.lower():
        result = flaky_search_tool(step)
    elif "data" in step.lower() or "get" in step.lower() or "retrieve" in step.lower():
        result = flaky_data_tool(step)
    elif "calcul" in step.lower() or "compute" in step.lower():
        result = flaky_calculate_tool(step)
    else:
        result = flaky_search_tool(step)
    
    # Сохраняем в историю
    history_entry = {"step": step, "step_index": step_idx, "result": result}
    new_history = state["execution_history"] + [history_entry]
    
    return {"execution_history": new_history}

def replan_observer_node(state: ReplanState):
    console.print("--- [REPLAN] OBSERVER: Анализ результата... ---")
    observer_llm = llm.with_structured_output(Observation)
    
    last_execution = state["execution_history"][-1]
    
    prompt = f"""
    Analyze this tool execution result:
    
    Step: {last_execution['step']}
    Result: {last_execution['result']}
    
    Task context: {state['user_request']}
    
    Analyze the result and provide:
    1. Status: success (got what we needed), partial (got some data), failure (error), needs_alternative (need different approach)
    2. Any facts or data that can be extracted from this result
    3. Any issues that were encountered
    4. Suggestions for improving the plan if needed
    
    Look for keywords like ERROR, PARTIAL_DATA, STALE_DATA, TIMEOUT, FORMAT_ERROR, CLARIFICATION_NEEDED.
    """
    
    try:
        observation = observer_llm.invoke(prompt)
    except Exception as e:
        # Fallback если не удалось распарсить
        observation = Observation(
            status="partial",
            extracted_facts=[last_execution['result'][:200]],
            issues_found=["Parsing error in observation"],
            suggestions=[]
        )
    
    status_emoji = {"success": "✅", "partial": "⚠️", "failure": "❌", "needs_alternative": "🔄"}
    console.print(f"--- [REPLAN] OBSERVER: Статус: {status_emoji.get(observation.status, '?')} {observation.status} ---")
    
    if observation.extracted_facts:
        console.print(f"[green]Извлечено фактов: {len(observation.extracted_facts)}[/green]")
    if observation.issues_found:
        console.print(f"[yellow]Проблемы: {observation.issues_found}[/yellow]")
    
    # Обновляем состояние
    new_facts = state["extracted_facts"] + observation.extracted_facts
    new_observations = state.get("observations", []) + [observation.model_dump_json()]
    
    # Обновляем историю с наблюдением
    updated_history = state["execution_history"].copy()
    updated_history[-1]["observation"] = observation.model_dump()
    
    return {
        "execution_history": updated_history,
        "observations": new_observations,
        "extracted_facts": new_facts
    }

def replan_replanner_node(state: ReplanState):
    console.print("--- [REPLAN] REPLANNER: Принятие решения... ---")
    replanner_llm = llm.with_structured_output(ReplanDecision)
    
    last_observation = state["execution_history"][-1].get("observation", {})
    remaining_steps = state["current_plan"][state["current_step_index"] + 1:]
    
    prompt = f"""
    Decide on the next action based on the execution history.
    
    User request: {state['user_request']}
    
    Current step completed: {state['current_step_index'] + 1}/{len(state['current_plan'])}
    Last observation: {json.dumps(last_observation)}
    
    Remaining steps in plan: {remaining_steps}
    Extracted facts so far: {state['extracted_facts']}
    
    Replan count: {state['replan_count']}/{state['max_replans']}
    
    Choose action:
    - "continue": Move to next step in current plan
    - "modify_plan": Modify remaining steps based on observations
    - "full_replan": Create entirely new plan (use sparingly, counts toward limit)
    - "synthesize": Enough data collected, generate final answer
    
    If modifying or replanning, provide the new steps.
    """
    
    try:
        decision = replanner_llm.invoke(prompt)
    except Exception as e:
        # Fallback
        decision = ReplanDecision(
            action="continue",
            reasoning="Fallback to continue"
        )
    
    action_emoji = {"continue": "➡️", "modify_plan": "📝", "full_replan": "🔄", "synthesize": "📊"}
    console.print(f"--- [REPLAN] REPLANNER: {action_emoji.get(decision.action, '?')} {decision.action} ---")
    console.print(f"[dim]Причина: {decision.reasoning}[/dim]")
    
    if decision.action == "continue":
        return {"current_step_index": state["current_step_index"] + 1}
    
    elif decision.action == "modify_plan":
        if decision.new_steps:
            # Заменяем оставшиеся шаги
            new_plan = state["current_plan"][:state["current_step_index"] + 1] + decision.new_steps
            console.print(Panel("\n".join([f"{i+1}. {s}" for i, s in enumerate(new_plan)]), title="Модифицированный план", border_style="yellow"))
            return {
                "current_plan": new_plan,
                "current_step_index": state["current_step_index"] + 1
            }
        return {"current_step_index": state["current_step_index"] + 1}
    
    elif decision.action == "full_replan":
        return {
            "replan_count": state["replan_count"] + 1,
            "current_step_index": 0
        }
    
    else:  # synthesize
        return {}  # Handled by router

def replan_synthesizer_node(state: ReplanState):
    console.print("--- [REPLAN] SYNTHESIZER: Генерация ответа... ---")
    
    facts = "\n".join([f"- {fact}" for fact in state["extracted_facts"]])
    
    prompt = f"""
    Generate a comprehensive answer.
    
    User request: {state['user_request']}
    
    Extracted facts:
    {facts}
    
    Execution summary:
    - Total steps executed: {len(state['execution_history'])}
    - Replanning events: {state['replan_count']}
    
    Provide a comprehensive answer. If some data was incomplete or had issues, acknowledge the limitations.
    """
    
    answer = llm.invoke(prompt).content
    return {"final_answer": answer}

def replan_router(state: ReplanState):
    # Проверяем последнее решение replanner'а
    last_execution = state["execution_history"][-1] if state["execution_history"] else {}
    last_observation = last_execution.get("observation", {})
    
    # Проверка лимита перепланирований
    if state["replan_count"] >= state["max_replans"]:
        console.print("--- [REPLAN] ROUTER: Лимит перепланирований, финализация ---")
        return "synthesize"
    
    # Если нужно полное перепланирование
    if state["current_step_index"] == 0 and state["replan_count"] > 0:
        return "plan"
    
    # Если все шаги выполнены
    if state["current_step_index"] >= len(state["current_plan"]):
        return "synthesize"
    
    return "execute"

# Построение графа Replan
replan_graph = StateGraph(ReplanState)
replan_graph.add_node("plan", replan_planner_node)
replan_graph.add_node("execute", replan_executor_node)
replan_graph.add_node("observe", replan_observer_node)
replan_graph.add_node("replan", replan_replanner_node)
replan_graph.add_node("synthesize", replan_synthesizer_node)

replan_graph.set_entry_point("plan")
replan_graph.add_edge("plan", "execute")
replan_graph.add_edge("execute", "observe")
replan_graph.add_edge("observe", "replan")
replan_graph.add_conditional_edges("replan", replan_router, {
    "plan": "plan",
    "execute": "execute",
    "synthesize": "synthesize"
})
replan_graph.add_edge("synthesize", END)

replan_agent = replan_graph.compile()
print("Replan Dynamic агент скомпилирован.")

## Фаза 3: Сравнение агентов

Теперь запустим обоих агентов на одной и той же сложной задаче, которая потребует адаптации к сбоям инструментов.

### Шаг 3.1: Тестирование и Сравнение

**Что мы будем делать:**
1. Определим сложный запрос с множеством шагов, который гарантированно вызовет сбои.
2. Запустим PEV агента и посмотрим, как он справится (ожидается повтор шагов).
3. Запустим Replan агента и увидим адаптацию стратегии.
4. Используем `ComparisonResult` для автоматической оценки качества работы обоих агентов.

In [ ]:
test_query = """Analyze TechCorp company: 
1. Find their main competitors and market share
2. Get their revenue data for the last fiscal year
3. Calculate revenue per employee
4. Provide an overall assessment"""

console.print(Panel(test_query, title="Тестовый запрос", border_style="magenta"))

In [ ]:
console.print("\n" + "="*60)
console.print("[bold blue]Тест 1: Базовый PEV агент[/bold blue]")
console.print("="*60 + "\n")

reset_tool_counters()

pev_result = pev_agent.invoke({
    "user_request": test_query,
    "plan": None,
    "current_step_index": 0,
    "last_result": None,
    "collected_data": [],
    "final_answer": None,
    "retry_count": 0,
    "max_retries": 2
}, {"recursion_limit": 30})

console.print("\n[bold]Результат PEV агента:[/bold]")
console.print(Panel(Markdown(pev_result["final_answer"]), border_style="blue"))

In [ ]:
console.print("\n" + "="*60)
console.print("[bold green]Тест 2: Replan Dynamic агент[/bold green]")
console.print("="*60 + "\n")

reset_tool_counters()

replan_result = replan_agent.invoke({
    "user_request": test_query,
    "current_plan": [],
    "current_step_index": 0,
    "execution_history": [],
    "observations": [],
    "extracted_facts": [],
    "final_answer": None,
    "replan_count": 0,
    "max_replans": 3
}, {"recursion_limit": 40})

console.print("\n[bold]Результат Replan агента:[/bold]")
console.print(Panel(Markdown(replan_result["final_answer"]), border_style="green"))

### Сравнительный анализ

In [ ]:
# Оценка с помощью LLM-as-Judge
class ComparisonResult(BaseModel):
    pev_score: int = Field(description="Оценка PEV агента (1-10)")
    replan_score: int = Field(description="Оценка Replan агента (1-10)")
    pev_strengths: List[str] = Field(description="Сильные стороны PEV")
    replan_strengths: List[str] = Field(description="Сильные стороны Replan")
    winner: str = Field(description="Победитель: PEV или Replan")
    justification: str = Field(description="Обоснование")

judge_llm = llm.with_structured_output(ComparisonResult)

judge_prompt = f"""
Compare these two agent outputs for the same task.
Task: {test_query}
PEV Agent Output: {pev_result['final_answer']}
Replan Agent Output: {replan_result['final_answer']}
"""

comparison = judge_llm.invoke(judge_prompt)

console.print(Panel(
    f"[bold]Winner:[/bold] {comparison.winner}\n\n"
    f"[bold]PEV Score:[/bold] {comparison.pev_score}/10\n"
    f"[bold]Replan Score:[/bold] {comparison.replan_score}/10\n\n"
    f"[bold]Justification:[/bold] {comparison.justification}",
    title="LLM Judge Comparison",
    border_style="gold1"
))